<a href="https://colab.research.google.com/github/LleilaA13/Thesis-MUL/blob/random-data-forgetting/notebooks/10_random_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### This notebook uses the Lucent library, which is licensed under the Apache License, Version 2.0 (the "License");

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Feature visualization: Pre vs. Post Machine Unlearning
This notebook demonstrates how machine unlearning affects the internal representations of neural networks by visualizing learned features before and after the unlearning process. We employ the **Lucent** library to generate feature visualizations and compare activation patterns across different layers of the network.

**Feature visualization:**
Generally speaking, neural networks are differentiable with respect to their inputs, meaning that certain inputs cause a certain behaviour.

Starting with random noise, we optimize an image to activate a particular neuron.



**How Lucent works:**
The core idea is to optimize the input image to the network such that a certain neuron or channel gets maximally excited.


**Random Forgetting with SalUn: Resnet-50 on Tiny ImageNet:**

We randomly select and forget **10% of the training data** and provide a comprehensive analysis of how unlearning affects different model components.


## Install, Import, Load model

In [2]:
!pip install --quiet git+https://github.com/greentfrapp/lucent.git

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.8/252.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 68.5 MB/s eta 0:00:00


In [3]:
import torch
from torchvision import models
import matplotlib.pyplot as plt
import torchvision.transforms as T
from lucent.modelzoo.util import get_model_layers
from lucent.optvis import render, param, transform, objectives

In [4]:
#fix seed for reproducibility
torch.manual_seed(0)

In [7]:

def load_resnet50(path):
  model = models.resnet50(num_classes=200)
  checkpoint = torch.load(path, map_location='cpu', weights_only=False)
  try:
      model.load_state_dict(checkpoint['state_dict'], strict=False)
  except:
      model.load_state_dict(checkpoint, strict=False)
  model.eval()
  return model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Replace with the actual paths to your models
pretrained_model_path = '/content/sample_pretrained_model.pth.tar'
unlearned_model_path = '/content/sample_unlearned_model.pth.tar'

# Load the pretrained and unlearned models
pretrained_model = load_resnet50(path=pretrained_model_path)
unlearned_model = load_resnet50(path=unlearned_model_path)

print("Pretrained model loaded successfully.")
print("Unlearned model loaded successfully.")

# Weight Analysis:

First of all, we need to show the most influcend weights, neurons, layers and channels.

**Objectives:**

What loss function do we want to minimize?

Or from another point of view, what part of the model do we want to understand?

We are trying to generate an image that causes a particular neuron or filter to activate strongly.
